# Healthcare Lab 3: Clinical Trial Assistant -- Protocol Analysis and Eligibility

**Series**: Agentic AI Science Playbook -- Healthcare Domain
**Goal**: Build an agent that analyzes clinical trial protocols and checks patient eligibility against inclusion/exclusion criteria.
**Prerequisites**: HC Labs 1-2, Foundation Labs 0-4.
**Time**: ~60 min.

---

## Background: Clinical Trials

Clinical trials follow strict protocols defining:
- **Phase**: I (safety) → II (efficacy signals) → III (large-scale efficacy) → IV (post-market)
- **Eligibility criteria**: Who can and cannot participate (inclusion and exclusion criteria)
- **Primary endpoint**: The main outcome the trial is designed to measure
- **Randomization and blinding**: Single-blind, double-blind, open-label

Manually checking patient eligibility against dozens of criteria is time-consuming and error-prone. An AI agent can automate initial screening while flagging cases needing physician review.

## Learning Objectives

By the end of this lab, you will be able to:
- Build agents that screen patients against clinical trial eligibility criteria
- Implement audience-adaptive protocol explanations
- Handle the three-outcome pattern: ELIGIBLE / INELIGIBLE / REQUIRES_REVIEW
- Design agents for safety-critical decision support

## Why This Matters

Clinical trials struggle with enrollment — only 3-5% of eligible patients are enrolled. A major bottleneck is the manual screening process: coordinators must check each patient against dozens of inclusion/exclusion criteria.

AI-assisted screening can **10x the throughput** of eligibility checking while maintaining accuracy, connecting more patients with potentially life-saving treatments.

> **Critical principle**: The agent produces a preliminary screen. **ALL enrollment decisions require physician review.** The agent's job is to flag clearly ineligible patients and highlight uncertain cases for human attention.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | HC Labs 1-2, Foundation Labs 0-4 |

**NVIDIA Connection**: Clinical trial AI is one of the fastest-growing applications of NVIDIA's healthcare platform. IQVIA's partnership with NVIDIA uses NIM-powered agents to automate clinical data review, reducing timeline from 7 weeks to 2 weeks. The eligibility checking pattern you build here is the same agent architecture.

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field
from typing import Literal

## Simulated Trial Protocol

### Load a Simulated Clinical Trial Protocol
A Phase III diabetes trial with 6 inclusion criteria and 10 exclusion criteria. This is the protocol our agent will use for eligibility screening. It includes age ranges, lab value thresholds, and medication restrictions typical of a GLP-1 agonist trial.

In [ ]:
SAMPLE_PROTOCOL = {
    "trial_id": "NCT-SAMPLE-001",
    "title": "A Phase III Randomized Trial of Novel GLP-1 Agonist vs. Placebo in Type 2 Diabetes with Cardiovascular Risk",
    "phase": "III",
    "sponsor": "BioPharma Corp",
    "primary_endpoint": "3-point MACE (cardiovascular death, non-fatal MI, non-fatal stroke) at 3 years",
    "inclusion_criteria": [
        "Age 40-80 years",
        "Type 2 diabetes mellitus diagnosis >= 6 months",
        "HbA1c >= 7.0% and <= 10.0% at screening",
        "At least one cardiovascular risk factor (hypertension, dyslipidemia, smoking, obesity BMI>30)",
        "eGFR >= 30 mL/min/1.73m2",
        "Willing and able to provide written informed consent",
    ],
    "exclusion_criteria": [
        "Type 1 diabetes mellitus",
        "Personal or family history of medullary thyroid carcinoma",
        "Personal or family history of multiple endocrine neoplasia type 2 (MEN2)",
        "Current use of GLP-1 receptor agonists or DPP-4 inhibitors",
        "eGFR < 30 mL/min/1.73m2",
        "Active malignancy in past 5 years",
        "Pregnancy or planning pregnancy",
        "Severe gastroparesis",
        "Recent MI or stroke within 3 months of screening",
        "Pancreatitis history",
    ],
    "primary_outcome_timepoint": "36 months",
}
print(f"Protocol loaded: {SAMPLE_PROTOCOL['title'][:60]}")
print(f"Inclusion criteria: {len(SAMPLE_PROTOCOL['inclusion_criteria'])}")
print(f"Exclusion criteria: {len(SAMPLE_PROTOCOL['exclusion_criteria'])}")

## Concept: The Three-Outcome Pattern

Unlike most classification tasks that produce binary yes/no, clinical eligibility requires THREE outcomes:

| Outcome | Meaning | Agent Action |
|---------|---------|-------------|
| **ELIGIBLE** | All inclusion met, no exclusion triggered | Recommend for enrollment discussion |
| **INELIGIBLE** | Clear exclusion criterion triggered | Do not proceed |
| **REQUIRES_REVIEW** | Insufficient information or borderline values | Flag for physician review |

The `REQUIRES_REVIEW` outcome is the most important — it prevents the agent from making decisions when it shouldn't. This is the **human-in-the-loop** pattern from Foundation principles, applied to a safety-critical domain.

### Define Trial Assistant Tool Schemas
Three tools: `check_eligibility` screens patients, `explain_criterion` clarifies medical terms, and `summarize_protocol` adapts content for different audiences. Each tool has parameters for patient data, criterion text, or audience type.

## Tool Schemas

In [ ]:
class CheckEligibilityArgs(BaseModel):
    patient_summary: str = Field(..., description="Clinical summary of the patient including demographics, diagnoses, medications, labs.")
    protocol_id: str = Field(..., description="Trial protocol identifier.")

class ExplainCriterionArgs(BaseModel):
    criterion_text: str = Field(..., description="The eligibility criterion to explain.")
    clinical_context: str | None = Field(None, description="Clinical context for the explanation.")

class SummarizeProtocolArgs(BaseModel):
    protocol_section: str = Field("all", description="Section to summarize: all | endpoints | eligibility | design.")
    audience: str = Field("patient", description="Target audience: patient | physician | researcher.")

OPENAI_TOOLS = [
    {"type": "function", "function": {
        "name": "check_eligibility",
        "description": "Check if a patient meets the inclusion/exclusion criteria for a clinical trial.",
        "parameters": CheckEligibilityArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "explain_criterion",
        "description": "Explain what a specific eligibility criterion means in plain language.",
        "parameters": ExplainCriterionArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "summarize_protocol",
        "description": "Summarize a clinical trial protocol for a specific audience.",
        "parameters": SummarizeProtocolArgs.model_json_schema()}},
]
SCHEMA_MAP = {
    "check_eligibility": CheckEligibilityArgs,
    "explain_criterion": ExplainCriterionArgs,
    "summarize_protocol": SummarizeProtocolArgs,
}
print("Clinical trial tools:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Tool Implementations

### Implement Trial Tools
The eligibility checker sends the full protocol to the LLM and asks it to evaluate each criterion. The explainer translates medical jargon into plain language. The protocol summarizer adapts content depth based on target audience (patient, physician, or researcher).

In [ ]:
def execute_check_eligibility(args: CheckEligibilityArgs) -> str:
    inclusion = "\n".join(f"  {i+1}. {c}" for i, c in enumerate(SAMPLE_PROTOCOL["inclusion_criteria"]))
    exclusion = "\n".join(f"  {i+1}. {c}" for i, c in enumerate(SAMPLE_PROTOCOL["exclusion_criteria"]))
    prompt = (
        f"Patient summary: {args.patient_summary}\n\n"
        f"Trial: {SAMPLE_PROTOCOL['title']}\n"
        f"Inclusion criteria:\n{inclusion}\n"
        f"Exclusion criteria:\n{exclusion}\n\n"
        "For each criterion, state whether the patient MEETS, DOES NOT MEET, or UNCLEAR (insufficient info). "
        "Then give overall eligibility: ELIGIBLE, INELIGIBLE, or REQUIRES_REVIEW.\n"
        "Return JSON: {\"inclusion_checks\": [{\"criterion\": ..., \"status\": ..., \"reasoning\": ...}], "
        "\"exclusion_checks\": [{\"criterion\": ..., \"status\": ..., \"reasoning\": ...}], "
        "\"overall_eligibility\": ..., \"notes\": ...}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=1500,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

def execute_explain_criterion(args: ExplainCriterionArgs) -> str:
    ctx = f" Context: {args.clinical_context}" if args.clinical_context else ""
    prompt = (
        f"Explain this clinical trial eligibility criterion in plain, accessible language:{ctx}\n"
        f"Criterion: {args.criterion_text}\n"
        "Include: what it means, why it matters for trial safety/validity, and any common questions."
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.3, max_tokens=300,
        messages=[{"role": "user", "content": prompt}])
    return (r.choices[0].message.content or "").strip()

def execute_summarize_protocol(args: SummarizeProtocolArgs) -> str:
    sections = {
        "all": json.dumps(SAMPLE_PROTOCOL, indent=2)[:2000],
        "eligibility": f"Inclusion: {SAMPLE_PROTOCOL['inclusion_criteria']}\nExclusion: {SAMPLE_PROTOCOL['exclusion_criteria']}",
        "endpoints": f"Primary: {SAMPLE_PROTOCOL['primary_endpoint']}",
        "design": f"Phase {SAMPLE_PROTOCOL['phase']}, {SAMPLE_PROTOCOL['primary_outcome_timepoint']} follow-up",
    }
    content = sections.get(args.protocol_section, sections["all"])
    prompt = (
        f"Summarize this clinical trial protocol for a {args.audience}:\n{content}\n"
        f"Use appropriate language for {args.audience}. Be concise and accurate."
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.3, max_tokens=400,
        messages=[{"role": "user", "content": prompt}])
    return (r.choices[0].message.content or "").strip()

def run_tool(name, args):
    if name == "check_eligibility": return execute_check_eligibility(args)
    if name == "explain_criterion": return execute_explain_criterion(args)
    if name == "summarize_protocol": return execute_summarize_protocol(args)
    return f"[error] Unknown: {name}"

TRIAL_SYSTEM = (
    "You are a clinical trial assistant. Help with patient eligibility screening, "
    "protocol explanation, and trial design questions. Use the provided tools. "
    "Always note that final enrollment decisions require physician review."
)

def trial_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": TRIAL_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "content": msg.content}
    call = msg.tool_calls[0]
    validated = SCHEMA_MAP[call.function.name](**json.loads(call.function.arguments))
    return {"tool": call.function.name, "args": validated}

## Experiment 1: Eligible Patient

### Experiment 1: Screen an Eligible Patient
A 62-year-old male who meets all inclusion criteria and triggers no exclusions. The agent should classify him as ELIGIBLE. The patient has T2DM for 8 years, HbA1c 8.4%, adequate kidney function, and no disqualifying conditions.

In [ ]:
patient_eligible = (
    "62-year-old male with T2DM x 8 years, HbA1c 8.4%, eGFR 58 mL/min/1.73m2. "
    "Hypertension and dyslipidemia. Current medications: metformin 1000mg BID, lisinopril 10mg. "
    "No history of thyroid cancer or MEN2. No GLP-1 or DPP-4 use. Non-smoker, BMI 31. "
    "No recent MI or stroke. No malignancy history. Not pregnant."
)

r = trial_agent(f"Check this patient's eligibility for trial {SAMPLE_PROTOCOL['trial_id']}: {patient_eligible}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print(f"OVERALL ELIGIBILITY: {result.get('overall_eligibility','?')}")
    print(f"Notes: {result.get('notes','')}")
    print("\nInclusion checks:")
    for check in result.get("inclusion_checks", []):
        print(f"  [{check.get('status','?')}] {check.get('criterion','?')[:60]}")
    print("\nExclusion checks:")
    for check in result.get("exclusion_checks", []):
        print(f"  [{check.get('status','?')}] {check.get('criterion','?')[:60]}")

<details>
<summary>Expected output (click to expand)</summary>

```
OVERALL ELIGIBILITY: ELIGIBLE
Notes: Patient meets all inclusion criteria and no exclusion criteria are triggered.

Inclusion checks:
  [MEETS] Age 40-80 years
  [MEETS] Type 2 diabetes mellitus diagnosis >= 6 months
  [MEETS] HbA1c >= 7.0% and <= 10.0% at screening
  [MEETS] At least one cardiovascular risk factor
  [MEETS] eGFR >= 30 mL/min/1.73m2
  [MEETS] Willing and able to provide written informed consent

Exclusion checks:
  [MEETS] Type 1 diabetes mellitus
  [MEETS] Personal or family history of medullary thyroid carcinoma
  [MEETS] Current use of GLP-1 receptor agonists or DPP-4 inhibitors
  ...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. "MEETS" for exclusion criteria means the patient does NOT have the exclusion condition (i.e., they pass the check).
</details>

## Experiment 2: Ineligible Patient

### Experiment 2: Screen an Ineligible Patient
A patient with multiple exclusion triggers (low eGFR, GLP-1 use, thyroid cancer history, pancreatitis). Watch the agent identify each exclusion criterion that this patient fails. The agent should provide specific reasoning for each failed criterion.

In [ ]:
patient_ineligible = (
    "55-year-old female with T2DM x 3 years, HbA1c 7.8%, eGFR 22 mL/min/1.73m2. "
    "Currently on dulaglutide (GLP-1 agonist) and metformin. "
    "History of papillary thyroid carcinoma (in remission 2 years). "
    "History of acute pancreatitis 18 months ago. BMI 28."
)

r = trial_agent(f"Is this patient eligible for trial {SAMPLE_PROTOCOL['trial_id']}? {patient_ineligible}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print(f"OVERALL ELIGIBILITY: {result.get('overall_eligibility','?')}")
    print(f"Notes: {result.get('notes','')}")
    print("\nKey exclusion issues:")
    for check in result.get("exclusion_checks", []):
        if check.get("status") == "DOES NOT MEET":
            print(f"  EXCLUDED: {check.get('criterion','?')[:60]}")
            print(f"    Reason: {check.get('reasoning','?')[:80]}")

<details>
<summary>Expected output (click to expand)</summary>

```
OVERALL ELIGIBILITY: INELIGIBLE
Notes: Patient has multiple exclusion criteria triggered.

Key exclusion issues:
  EXCLUDED: eGFR < 30 mL/min/1.73m2
    Reason: Patient eGFR is 22 mL/min/1.73m2, below the 30 threshold
  EXCLUDED: Current use of GLP-1 receptor agonists or DPP-4 inhibitors
    Reason: Patient is currently on dulaglutide (a GLP-1 agonist)
  EXCLUDED: Pancreatitis history
    Reason: Patient has history of acute pancreatitis 18 months ago
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Experiment 3: Protocol Summary for Different Audiences

### Experiment 3: Audience-Adapted Protocol Summary
The same protocol, summarized for three audiences: patient (simple), physician (clinical), and researcher (detailed). This demonstrates how the agent adjusts vocabulary, technical depth, and emphasis depending on who will read the summary.

In [ ]:
for audience in ["patient", "physician", "researcher"]:
    r = trial_agent(f"Summarize the eligibility criteria of this trial for a {audience}.")
    if r["tool"]:
        result = run_tool(r["tool"], r["args"])
        print(f"\n=== For {audience.upper()} ===")
        print(result[:400])

<details>
<summary>Expected output (click to expand)</summary>

```
=== For PATIENT ===
This study is testing a new diabetes medicine to see if it can help protect
your heart. To join, you need to be between 40 and 80 years old, have had
type 2 diabetes for at least 6 months, and have certain heart risk factors...

=== For PHYSICIAN ===
Phase III RCT evaluating a novel GLP-1 agonist vs. placebo in T2DM patients
with CV risk. Inclusion: age 40-80, T2DM >=6mo, HbA1c 7.0-10.0%, >=1 CV
risk factor, eGFR >=30. Key exclusions: T1DM, MTC/MEN2 history, current
GLP-1/DPP-4 use, active malignancy, recent MI/stroke (<3mo)...

=== For RESEARCHER ===
Phase III double-blind RCT (NCT-SAMPLE-001). Primary endpoint: 3-point MACE
at 36 months. Power calculation assumes 14% event rate in placebo arm.
Eligibility designed to enrich for CV events while excluding high-risk
populations (eGFR<30, recent ACS)...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **The agent flagged an ineligible patient on 3 criteria.** In practice, should the agent stop checking after the first exclusion, or check all criteria? Why?
2. **Protocol summaries adapted for patient vs. physician audiences.** What information is appropriate for patients but NOT for public posting? What privacy considerations arise?
3. **How would you evaluate** this agent's accuracy? What would a false positive (declaring ineligible patient eligible) vs. false negative cost in the real world?

## Healthcare Track Complete!

| Lab | Skill | Foundation Pattern Used |
|-----|-------|------------------------|
| HC 1 | Clinical NLP extraction | Tool contracts (Lab 2), Safety checks |
| HC 2 | Evidence-based literature | Multi-step pipelines, PICO/GRADE |
| HC 3 | Trial eligibility | Three-outcome classification, Human-in-loop |

**Next**: Try the Bioinformatics domain track!

## Summary

| Tool | Capability |
|------|-----------|
| `check_eligibility` | Systematic inclusion/exclusion check with reasoning |
| `explain_criterion` | Plain-language criterion explanation |
| `summarize_protocol` | Audience-adapted protocol summaries |

**Key design principles**:
- Each eligibility check includes status + reasoning (not just yes/no)
- `REQUIRES_REVIEW` status flags uncertain cases for physician review
- Audience adaptation adjusts technical depth

**Completed Healthcare Domain track!** Move to Bioinformatics domain.

---
*Agentic AI Science Playbook -- Healthcare Domain, Lab HC3.*

**Disclaimer**: This agent is for educational purposes. All eligibility decisions require qualified physician review.